**Importing Libraries**

In [10]:
import pandas as pd
import numpy as np

**Preview data**

In [11]:
df = pd.read_csv('student_performance_data.csv')

print("Initial Data:")
display(df.head())

print("\nDataset Info:")
print(df.info())


Initial Data:


,student_id,gender,study_hours_per_day,attendance_percentage,assignment_score,midterm_score,final_exam_score,participation_score,internet_access,extra_classes,parent_education,sleep_hours,overall_score,grade
0,100000,Male,4.54,69.98,36.47,70.70,53.10,17.96,Yes,No,Master,8.09,52.3480,D
1,100001,Female,5.26,84.80,34.25,27.92,87.17,11.29,No,Yes,Bachelor,4.73,53.9485,D
2,100002,Male,8.69,73.76,72.29,70.92,99.61,76.10,No,Yes,PhD,8.73,82.0375,B
3,100003,Male,4.06,45.00,97.63,31.73,88.85,33.55,No,No,Bachelor,8.22,66.4110,C
4,100004,Male,8.83,51.13,65.19,78.28,54.23,88.99,No,No,Bachelor,8.59,65.6005,C



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             10000 non-null  int64  
 1   gender                 10000 non-null  object 
 2   study_hours_per_day    10000 non-null  float64
 3   attendance_percentage  10000 non-null  float64
 4   assignment_score       10000 non-null  float64
 5   midterm_score          10000 non-null  float64
 6   final_exam_score       10000 non-null  float64
 7   participation_score    10000 non-null  float64
 8   internet_access        10000 non-null  object 
 9   extra_classes          10000 non-null  object 
 10  parent_education       10000 non-null  object 
 11  sleep_hours            10000 non-null  float64
 12  overall_score          10000 non-null  float64
 13  grade                  10000 non-null  object 
dtypes: float64(8), int64(1), object(5)
memor

**Creating Cleaning & Preprocessing Pipeline**

In [12]:
def clean_data(df):
    # Remove Duplicates
    df = df.drop_duplicates()

    # Handle Missing Values
    # Separate numeric and categorical columns
    num_cols = df.select_dtypes(include=np.number).columns
    cat_cols = df.select_dtypes(exclude=np.number).columns

    # Fill numeric columns with median
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    # Fill categorical columns with mode
    for col in cat_cols:
        if df[col].mode().empty == False:
            df[col] = df[col].fillna(df[col].mode()[0])

    # Fix Data Formats
    # Convert possible numeric columns stored as strings
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except:
            pass

    # Convert date columns if any
    for col in df.columns:
        if 'date' in col.lower():
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # Handle Outliers (Optional)
    for col in num_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = np.clip(df[col], lower, upper)

    return df


**Applying Cleaning Pipeline**

In [13]:
df_cleaned = clean_data(df)

print("\nCleaned Data:")
display(df_cleaned.head())

# Encoding & Scaling
from sklearn.preprocessing import LabelEncoder, StandardScaler

def preprocess_data(df):

    df = df.copy()

    # Encode categorical variables
    label_encoders = {}
    cat_cols = df.select_dtypes(exclude=np.number).columns

    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    # Scale numeric features
    scaler = StandardScaler()
    num_cols = df.select_dtypes(include=np.number).columns
    df[num_cols] = scaler.fit_transform(df[num_cols])

    return df, label_encoders, scaler


df_processed, encoders, scaler = preprocess_data(df_cleaned)

print("\nProcessed Data (Ready for ML):")
display(df_processed.head())



Cleaned Data:


,student_id,gender,study_hours_per_day,attendance_percentage,assignment_score,midterm_score,final_exam_score,participation_score,internet_access,extra_classes,parent_education,sleep_hours,overall_score,grade
0,100000,Male,4.54,69.98,36.47,70.70,53.10,17.96,Yes,No,Master,8.09,52.3480,D
1,100001,Female,5.26,84.80,34.25,27.92,87.17,11.29,No,Yes,Bachelor,4.73,53.9485,D
2,100002,Male,8.69,73.76,72.29,70.92,99.61,76.10,No,Yes,PhD,8.73,82.0375,B
3,100003,Male,4.06,45.00,97.63,31.73,88.85,33.55,No,No,Bachelor,8.22,66.4110,C
4,100004,Male,8.83,51.13,65.19,78.28,54.23,88.99,No,No,Bachelor,8.59,65.6005,C



Processed Data (Ready for ML):


,student_id,gender,study_hours_per_day,attendance_percentage,assignment_score,midterm_score,final_exam_score,participation_score,internet_access,extra_classes,parent_education,sleep_hours,overall_score,grade
0,-1.731878,0.997403,-0.357681,-0.029709,-1.408743,0.385888,-0.587678,-1.431081,0.987874,-1.017554,0.458884,1.086863,-1.122250,1.468102
1,-1.731531,-1.002603,-0.080117,0.827198,-1.519457,-1.584680,1.097693,-1.688067,-1.012275,0.982749,-1.332232,-1.217390,-0.965767,1.468102
2,-1.731185,0.997403,1.242163,0.188854,0.377632,0.396022,1.713074,0.808980,-1.012275,0.982749,1.354441,1.525769,1.780529,-1.230119
3,-1.730838,0.997403,-0.542723,-1.474079,1.641360,-1.409181,1.180799,-0.830418,-1.012275,-1.017554,-1.332232,1.176016,0.252707,0.118992
4,-1.730492,0.997403,1.296133,-1.119636,0.023549,0.735045,-0.531779,1.305616,-1.012275,-1.017554,-1.332232,1.429758,0.173463,0.118992


**Saved Cleaned Dataset**

In [14]:
# Save Cleaned Dataset
df_cleaned.to_csv('cleaned_student_data.csv', index=False)
df_processed.to_csv('processed_student_data.csv', index=False)

print("\nFiles saved successfully!")


Files saved successfully!
